# P16: QLoRA instruction-tuning of a small LLM for Hindi

Runs on a free GPU:
- **Colab:** Runtime → Change runtime type → T4 GPU.
- **Kaggle:** Settings → Accelerator → GPU T4, and turn **Internet on**.

The fixed train/val/test split is already in the repo (`data/splits/`). Don't regenerate it, so every run uses the same test prompts.

In [ ]:
!nvidia-smi

## Setup

In [ ]:
import os, sys, json, re

ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/sunny4992mishra/QLoRA-Instruction-DL.git"
REPO = ("/kaggle/working" if ON_KAGGLE else "/content") + "/QLoRA-Instruction-DL"

if not os.path.exists(REPO):
    !git clone {REPO_URL} {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}
!pip install -q -r requirements.txt

In [ ]:
# Save adapters and outputs where they survive a disconnect.
# Colab: Google Drive. Kaggle: /kaggle/working (kept when you "Save Version").
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE = "/content/drive/MyDrive/p16"
elif ON_KAGGLE:
    SAVE = "/kaggle/working/p16"
else:
    SAVE = "outputs"
os.makedirs(f"{SAVE}/runs", exist_ok=True)
os.makedirs(f"{SAVE}/results", exist_ok=True)

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
TAG = MODEL.split("/")[-1]
N_TRAIN = sum(1 for _ in open("data/splits/train.jsonl", encoding="utf-8"))
print(json.load(open("data/split_stats.json")))

## 1. Zero-shot baseline

Base model without fine-tuning, on the 150 held-out test prompts. Add `--limit 20` for a quick check first.
If the session dies, re-run the same cell: prompts already in the output file are skipped.

In [ ]:
!python src/generate.py --model {MODEL} --out {SAVE}/results/{TAG}_zeroshot.jsonl

## 2. QLoRA fine-tuning (one run)

`N = 0` uses all training rows. `N = 250` uses the first 250 (subsets are nested: 100 ⊂ 250 ⊂ 500).

In [ ]:
RANK, N = 16, 0
NAME = f"{TAG}_r{RANK}_n{N or N_TRAIN}"
RUN = f"{SAVE}/runs/{NAME}"
!python src/train.py --model {MODEL} --rank {RANK} --train-size {N} --output-dir {RUN}

In [ ]:
info = json.load(open(f"{RUN}/run_info.json"))
print({k: info[k] for k in ["trainable_params", "trainable_pct", "gpu_memory_peak_training",
                             "train_seconds", "final_train_loss", "final_eval_loss"]})
print("val loss per epoch:", [(round(h["epoch"], 2), round(h["eval_loss"], 4))
                              for h in info["log_history"] if "eval_loss" in h])

## 3. Generate test answers with the adapter

In [ ]:
!python src/generate.py --model {MODEL} --adapter {RUN}/adapter --out {SAVE}/results/{NAME}.jsonl

## 4. Quick sanity check

Share of answers written mostly in Devanagari, and one example side by side. This is only a sanity check; the real evaluation is the LLM judge + human ratings.

In [ ]:
def load(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")]

def devanagari_share(text):
    letters = [c for c in text if c.isalpha() or "\u0900" <= c <= "\u097F"]
    return sum("\u0900" <= c <= "\u097F" for c in letters) / max(1, len(letters))

for name in [f"{TAG}_zeroshot", NAME]:
    path = f"{SAVE}/results/{name}.jsonl"
    if os.path.exists(path):
        rows = load(path)
        hindi = sum(devanagari_share(r["output"]) > 0.5 for r in rows) / len(rows)
        print(f"{name}: {len(rows)} answers, {hindi:.0%} mostly Devanagari")

base, tuned = load(f"{SAVE}/results/{TAG}_zeroshot.jsonl"), load(f"{SAVE}/results/{NAME}.jsonl")
i = 0
print("PROMPT:\n", base[i]["instruction"], "\n\nZERO-SHOT:\n", base[i]["output"][:600],
      "\n\nFINE-TUNED:\n", tuned[i]["output"][:600], "\n\nREFERENCE:\n", base[i]["reference"][:600])

## 5. Experiment grid

Rank sweep at full data, then data-size sweep at a fixed rank. Each run saves to `SAVE`; re-running the cell skips runs that already have `run_info.json`.
A run cut off mid-training restarts from scratch, so split the grid across team members / sessions.

In [ ]:
GRID = [(r, 0) for r in (4, 8, 16, 32, 64)] + [(16, n) for n in (100, 250, 500)]

for rank, n in GRID:
    name = f"{TAG}_r{rank}_n{n or N_TRAIN}"
    run = f"{SAVE}/runs/{name}"
    if not os.path.exists(f"{run}/run_info.json"):
        !python src/train.py --model {MODEL} --rank {rank} --train-size {n} --output-dir {run}
    !python src/generate.py --model {MODEL} --adapter {run}/adapter --out {SAVE}/results/{name}.jsonl

In [ ]:
# Summary table: one row per finished run.
rows = []
for name in sorted(os.listdir(f"{SAVE}/runs")):
    path = f"{SAVE}/runs/{name}/run_info.json"
    if os.path.exists(path):
        i = json.load(open(path))
        rows.append({"run": name, "rank": i["args"]["rank"], "train_rows": i["train_tokens"]["rows"],
                     "rows_after_trunc": i["train_tokens"]["kept"],
                     "trainable_params": i["trainable_params"],
                     "peak_gpu_gb": i["gpu_memory_peak_training"].get("peak_allocated_gb"),
                     "train_min": round(i["train_seconds"] / 60, 1), "val_loss": i["final_eval_loss"]})
import pandas as pd
pd.DataFrame(rows)